# Lekcija 11 - Protokol med agentoma (A2A)


## Namestitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## What is the A2A Protocol?

The **Protokol Agent-to-Agent (A2A)** je odprt standard, ki omogoča AI agentom, da komunicirajo,
se odkrijejo in sodelujejo — tudi če so zgrajeni na različnih ogrodjih ali gostovani
pri različnih storitvah.

Key concepts:

- **Odkritje** – Agenti objavijo *Kartica agenta*, ki opisuje njihove zmožnosti, s čimer drugim agentom (ali orkestratorjem) olajšajo iskanje pravega specialista za nalogo.
- **Izmenjava sporočil** – Agenti izmenjujejo strukturirana sporočila prek skupnega protokola, zato lahko zahteva enega agenta razume in izpolni drug agent ne glede na njegovo notranjo implementacijo.
- **Življenjski cikel naloge** – A2A določa stanje, kot so *oddano*, *v teku*, *zaključeno* in *neuspešno*, kar orkestratorju omogoča popoln vpogled v to, kako napreduje delegirana naloga.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## Ustvarjanje specializiranih potovalnih agentov


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Sodelovanje več agentov prek delovnega toka

Povežemo tri agente v zaporedni delovni tok, ki odraža A2A posredovanje sporočil:

1. **CurrencyExchangeAgent** prejme zahtevo uporabnika in pripravi priporočila glede menjave valut.
2. **ActivityPlannerAgent** prejme obogaten kontekst in doda priporočila za aktivnosti.
3. **TravelManagerAgent** združi oba vnosa v končni potovalni povzetek.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Razumevanje A2A v produkciji

V produkcijskem okolju protokol A2A odpira zmogljive scenarije med storitvami:

| Zmožnost | Opis |
|---|---|
| **Interoperabilnost med ogrodji** | Agent, zgrajen v enem ogrodju, lahko delegira naloge agentu, zgrajenemu v katerem koli drugem ogrodju, združljivem z A2A, kar omogoča resnično interoperabilnost med organizacijami. |
| **Meje storitev** | Agenti so lahko v ločenih mikrostoritvah, oblačnih regijah ali celo v različnih organizacijah, pri čemer še vedno nemoteno sodelujejo. |
| **Dinamično odkrivanje** | Orkestrator lahko med izvajanjem poizve register Agent Card, da najde najbolj primernega specialista za dano podnalogo. |
| **Pretakanje & potisna obvestila** | A2A podpira Server-Sent Events (SSE) za posodobitve napredka v realnem času in potisna obvestila za dolgotrajne naloge. |

Delovni tok, ki smo ga zgradili zgoraj, je poenostavljena, v-procesna različica tega vzorca. V dejanski postavitvi bi vsak agent izpostavil HTTP endpoint, objavil Agent Card in komuniciral prek protokola A2A JSON-RPC.


## Povzetek

V tej lekciji ste se naučili:

1. **Kaj je protokol A2A** — odprt standard za odkrivanje, sporočanje,
   in upravljanje nalog.
2. **Kako ustvariti specializirane agente** — agenta Currency Exchange, agenta Activity Planner,
   in orkestratorja Travel Manager.
3. **Kako povezati agente v delovni tok** — z uporabo `WorkflowBuilder` za modeliranje zaporednega
   posredovanja sporočil med agenti.
4. **Kako A2A deluje v produkciji** — omogočanje sodelovanja med različnimi ogrodji in storitvami
   z dinamičnim odkrivanjem in pretočnimi posodobitvami.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Izjava o omejitvi odgovornosti**:
Ta dokument je bil preveden s storitvijo za prevajanje z umetno inteligenco [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v izvirnem jeziku velja za avtoritativni vir. Za ključne informacije priporočamo strokovni prevod, opravljen s strani profesionalnega prevajalca. Ne odgovarjamo za morebitne nesporazume ali napačne razlage, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
